### Pacotes importados

In [1]:
using LinearAlgebra
using Printf
using Plots

## Chapter 13: Quasi-Newton methods

### Algorithm 13.1: Quasi-Newton BFGS method

![image.png](attachment:f1a50ab8-a76c-4c2f-8eea-33e22482cdcd.png)

![image.png](attachment:389e06b0-0c1c-4e3a-af0c-9b48c6998501.png)

Example 5.8: $f(x_1,x_2) = \frac{1}{2} x_1^2 + x_1 \cos(x_2)$

In [2]:
# Algorithm 13.1: Quasi-Newton BFGS method
# Example 5.8: f(x1,x2) = 1/2*x1^2 + x1*cos(x2)

function f_ex58(x)
    return 0.5*x[1]^2 + x[1]*cos(x[2])
end

function grad_ex58(x)
    return [
        x[1] + cos(x[2]),
        -x[1]*sin(x[2])
    ]
end

function backtracking_search(f, grad_f, x, d; α=1e-4, β=0.5, t0=1.0)
    t = t0
    g = grad_f(x)

    while f(x + t*d) > f(x) + α*t*dot(g, d)
        t *= β
    end

    return t
end

function bfgs_method(f, grad_f, x0; eps=1e-6, max_iter=1000)
    n = length(x0)
    x = copy(x0)
    H = Matrix{Float64}(I, n, n)   # inverse Hessian approximation

    history = []

    for k in 1:max_iter
        g = grad_f(x)
        push!(history, (k, copy(x), f(x), norm(g)))

        if norm(g) < eps
            break
        end

        # Search direction
        d = -H*g

        # Step size
        t = backtracking_search(f, grad_f, x, d)

        # Update point
        x_new = x + t*d

        # BFGS update
        s = x_new - x
        y = grad_f(x_new) - g

        if dot(y, s) > 1e-10
            ρ = 1.0 / dot(y, s)
            V = Matrix{Float64}(I, n, n) - ρ*s*y'
            H = V*H*V' + ρ*(s*s')
        else
            H = Matrix{Float64}(I, n, n)
        end

        x = x_new
    end

    return x, f(x), history
end

# Initial point
x0 = [1.0, 1.0]

x_star, f_star, history = bfgs_method(f_ex58, grad_ex58, x0)

println("BFGS Method - Example 5.8")
println("Initial point: ", x0)
println("Solution x*: ", round.(x_star, digits=6))
println("f(x*): ", round(f_star, digits=6))
println("Gradient norm: ", round(norm(grad_ex58(x_star)), digits=8))
println("Iterations: ", length(history))

println("\nIteration history:")
for (k, xk, fk, gnorm) in history
    @printf("Iter %3d | x = [% .6f, % .6f] | f(x) = % .8f | ||grad|| = %.8f\n",
            k, xk[1], xk[2], fk, gnorm)
end

BFGS Method - Example 5.8
Initial point: [1.0, 1.0]
Solution x*: [-1.0, 0.0]
f(x*): -0.5
Gradient norm: 2.5e-7
Iterations: 15

Iteration history:
Iter   1 | x = [ 1.000000,  1.000000] | f(x) =  1.04030231 | ||grad|| = 1.75516512
Iter   2 | x = [-0.540302,  1.841471] | f(x) =  0.29043018 | ||grad|| = 0.96094182
Iter   3 | x = [-0.018901,  1.505278] | f(x) = -0.00105886 | ||grad|| = 0.05024503
Iter   4 | x = [-0.054389,  1.480616] | f(x) = -0.00341911 | ||grad|| = 0.06485736
Iter   5 | x = [-0.090058,  1.426448] | f(x) = -0.00889945 | ||grad|| = 0.10409607
Iter   6 | x = [-0.143848,  1.337326] | f(x) = -0.02293383 | ||grad|| = 0.16505202
Iter   7 | x = [-0.231355,  1.197381] | f(x) = -0.05763521 | ||grad|| = 0.25339518
Iter   8 | x = [-0.364798,  0.981969] | f(x) = -0.13606489 | ||grad|| = 0.35826400
Iter   9 | x = [-0.555386,  0.678606] | f(x) = -0.27811259 | ||grad|| = 0.41387575
Iter  10 | x = [-0.778448,  0.329985] | f(x) = -0.43345787 | ||grad|| = 0.30284378
Iter  11 | x = [-1.09093

#### the Rosenbrock problem

In [4]:
# BFGS method applied to the Rosenbrock problem

function rosenbrock(x)
    return (1 - x[1])^2 + 100*(x[2] - x[1]^2)^2
end

function grad_rosenbrock(x)
    return [
        -2*(1 - x[1]) - 400*x[1]*(x[2] - x[1]^2),
        200*(x[2] - x[1]^2)
    ]
end

# Initial point for the Rosenbrock function
x0 = [-1.2, 1.0]

x_star, f_star, history = bfgs_method(rosenbrock, grad_rosenbrock, x0; eps=1e-6, max_iter=1000)

println("BFGS Method - Rosenbrock Problem")
println("Initial point: ", x0)
println("Solution x*: ", round.(x_star, digits=6))
println("f(x*): ", round(f_star, digits=10))
println("Gradient norm: ", round(norm(grad_rosenbrock(x_star)), digits=8))
println("Iterations: ", length(history))

println("\nIteration history:")
for (k, xk, fk, gnorm) in history
    @printf("Iter %3d | x = [% .6f, % .6f] | f(x) = % .8f | ||grad|| = %.8f\n",
            k, xk[1], xk[2], fk, gnorm)
end

BFGS Method - Rosenbrock Problem
Initial point: [-1.2, 1.0]
Solution x*: [1.0, 1.0]
f(x*): 0.0
Gradient norm: 9.0e-8
Iterations: 35

Iteration history:
Iter   1 | x = [-1.200000,  1.000000] | f(x) =  24.20000000 | ||grad|| = 232.86768775
Iter   2 | x = [-0.989453,  1.085938] | f(x) =  5.10111266 | ||grad|| = 43.89852092
Iter   3 | x = [-0.772346,  0.570592] | f(x) =  3.20842357 | ||grad|| = 12.66424834
Iter   4 | x = [-0.741285,  0.518426] | f(x) =  3.12866102 | ||grad|| = 14.13741465
Iter   5 | x = [-0.463158,  0.141711] | f(x) =  2.67087014 | ||grad|| = 21.94176327
Iter   6 | x = [-0.497591,  0.267292] | f(x) =  2.28156829 | ||grad|| = 4.04620151
Iter   7 | x = [-0.381971,  0.133909] | f(x) =  1.92422723 | ||grad|| = 5.18453835
Iter   8 | x = [-0.304751,  0.058294] | f(x) =  1.82194897 | ||grad|| = 9.71632044
Iter   9 | x = [-0.261659,  0.036137] | f(x) =  1.69629477 | ||grad|| = 8.75765906
Iter  10 | x = [-0.123350,  0.001352] | f(x) =  1.28113325 | ||grad|| = 4.03439005
Iter  11 | 